### 00 - Instalação

In [ ]:
!pip install easyocr
!pip install -q open_clip_torch
!pip install -q rapidfuzz
!pip install -q openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00


### 01 - Imports

In [ ]:
import os
import re
import cv2
import torch
import open_clip
import pandas as pd

from PIL import Image
from pprint import pprint

from rapidfuzz import fuzz

import easyocr

In [ ]:
ocr_engine = easyocr.Reader(
    ['pt'],
    gpu=torch.cuda.is_available()
)

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [ ]:
print(type(ocr_engine))

<class 'easyocr.easyocr.Reader'>


In [ ]:
dir(ocr_engine)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'character',
 'converter',
 'cudnn_benchmark',
 'detect',
 'detect_network',
 'detection_models',
 'detector',
 'device',
 'download_enabled',
 'getChar',
 'getDetectorPath',
 'get_detector',
 'get_textbox',
 'initDetector',
 'lang_char',
 'model_lang',
 'model_storage_directory',
 'quantize',
 'readtext',
 'readtext_batched',
 'readtextlang',
 'recognition_models',
 'recognize',
 'recognizer',
 'setDetector',
 'setLanguageList',
 'setModelLanguage',
 'support_detection_network',
 'user_network_directory',
 'verbose']

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="laion2b_s34b_b79k"
)

clip_model = clip_model.to(device)

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

### 02 - Configuração

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Caminho raiz do projeto
ROOT = "/content/drive/MyDrive/Revisão Sistemática_Sports InfoVis/NOVO/Teste/img-prompt/imagens"

# Pasta das imagens de referência
REF_FOLDER = os.path.join(ROOT, "ref")

In [ ]:
TASKS = [
    "B-COMP",
    "B-ID",
    "L-COMP",
    "L-ID",
    "P-COMP",
    "P-ID"
]

CONDITIONS = [
    "Zero-shot",
    "Few-shot",
    "Hibrido",
    "VQL"
]

In [ ]:
CONFIG = {

    "B-COMP": {
        "reference": "barras_comparacao_final.png",
        "compare_values": True
    },

    "B-ID": {
        "reference": "barras_identificacao_final.png",
        "compare_values": True
    },

    "L-COMP": {
        "reference": "linhas_comparacao_final.png",
        "compare_values": False
    },

    "L-ID": {
        "reference": "linhas_identificacao_final.png",
        "compare_values": True
    },

    "P-COMP": {
        "reference": "pontos_comparacao_final.png",
        "compare_values": False
    },

    "P-ID": {
        "reference": "pontos_identificacao_final.png",
        "compare_values": False
    }

}

In [ ]:
VALID_EXTENSIONS = (
    ".png",
    ".jpg",
    ".jpeg"
)

In [ ]:
print("ROOT:", ROOT)
print("Referências:", len(CONFIG))
print("Tarefas:", TASKS)
print("Condições:", CONDITIONS)

ROOT: /content/drive/MyDrive/Revisão Sistemática_Sports InfoVis/NOVO/Teste/img-prompt/imagens
Referências: 6
Tarefas: ['B-COMP', 'B-ID', 'L-COMP', 'L-ID', 'P-COMP', 'P-ID']
Condições: ['Zero-shot', 'Few-shot', 'Hibrido', 'VQL']


### 03 - OCR

In [ ]:
def get_image_size(image_path):
    """
    Retorna largura e altura da imagem.
    """

    img = cv2.imread(image_path)

    if img is None:
        raise FileNotFoundError(
            f"Imagem não encontrada: {image_path}"
        )

    height, width = img.shape[:2]

    return width, height

In [ ]:
def center(bbox):
    """
    Calcula o centro de um bounding box.
    """

    xs = [point[0] for point in bbox]
    ys = [point[1] for point in bbox]

    return (
        sum(xs) / len(xs),
        sum(ys) / len(ys)
    )

In [ ]:
def extract_text(image_path):
    """
    Extrai os textos de uma imagem utilizando EasyOCR.
    """

    result = ocr_engine.readtext(image_path)

    extracted = []

    for item in result:

        extracted.append({

            "text": item[1],

            "confidence": float(item[2]),

            "bbox": item[0]

        })

    return extracted

In [ ]:
def preprocess_ocr(ocr_data, image_path):
    """
    Normaliza as coordenadas do OCR e adiciona
    informações auxiliares para o parser.
    """

    width, height = get_image_size(image_path)

    data = []

    for item in ocr_data:

        x, y = center(item["bbox"])

        data.append({

            "text": item["text"],

            "confidence": float(item["confidence"]),

            "bbox": item["bbox"],

            "x": x,

            "y": y,

            "x_norm": x / width,

            "y_norm": y / height,

            "width": width,

            "height": height

        })

    return data

In [ ]:
def process_ocr(image_path):

    raw = extract_text(image_path)

    return preprocess_ocr(
        raw,
        image_path
    )

### 04 - Parser

In [ ]:
def identify_title(ocr):
    """
    Identifica o título da visualização.
    """

    candidates = [
        item for item in ocr
        if item["y_norm"] < 0.05
    ]

    candidates = sorted(
        candidates,
        key=lambda x: x["x"]
    )

    title = " ".join(
        item["text"]
        for item in candidates
    )

    return title.strip()

In [ ]:
def identify_legend(ocr):
    """
    Identifica os itens da legenda.
    """

    legend = []

    for item in ocr:

        if item["x_norm"] > 0.85:

            text = item["text"]

            if text.lower() != "nomes das medidas":

                legend.append(text)

    return legend

In [ ]:
import re

def identify_categories(ocr):
    """
    Identifica as categorias do gráfico.
    """

    categories = []

    for item in ocr:

        text = item["text"]

        if item["y_norm"] < 0.90:
            continue

        if "setor" in text.lower():
            continue

        if "R$" in text:
            continue

        if re.search(r"\d", text):
            continue

        if len(text) < 5:
            continue

        categories.append(item)

    categories = sorted(
        categories,
        key=lambda x: x["x"]
    )

    return [
        item["text"]
        for item in categories
    ]

In [ ]:
def identify_values(ocr):
    """
    Identifica rótulos numéricos da visualização.
    """

    values = []

    pattern = r"R\$\s*[\d,.]+"

    for item in ocr:

        if re.search(pattern, item["text"]):

            values.append(item["text"])

    return values

In [ ]:
def identify_x_axis(ocr):

    candidatos = []

    for item in ocr:

        txt = item["text"].lower()

        if "setor" in txt:

            candidatos.append(item)

    if len(candidatos):

        return candidatos[0]["text"]

    return ""

In [ ]:
def parse_chart_components(ocr):
    """
    Organiza todos os componentes extraídos da visualização.
    """

    components = {

        "title": identify_title(ocr),

        "legend": identify_legend(ocr),

        "categories": identify_categories(ocr),

        "values": identify_values(ocr),

        "x_axis": identify_x_axis(ocr)

    }

    return components

### 05 - Similaridade OCR

In [ ]:
def compare_text(ref, pred):

    if ref is None and pred is None:
        return 1.0

    if ref is not None and pred is None:
        return 0.0

    if ref is None and pred is not None:
        return 0.0

    return fuzz.ratio(ref, pred)/100


def compare_list(list1, list2):

    if len(list1) == 0 and len(list2) == 0:
        return 1.0

    if len(list1) == 0 or len(list2) == 0:
        return 0.0

    scores = []

    max_len = max(len(list1), len(list2))

    for i in range(max_len):

        if i >= len(list1) or i >= len(list2):
            scores.append(0)
        else:
            scores.append(
                fuzz.ratio(list1[i], list2[i]) / 100
            )

    return sum(scores) / len(scores)


def calculate_text_score(reference, generated, compare_values=True):

    scores={}

    scores["title"]=compare_text(
        reference["title"],
        generated["title"]
    )

    scores["x_axis"]=compare_text(
        reference["x_axis"],
        generated["x_axis"]
    )

    scores["legend"]=compare_list(
        reference["legend"],
        generated["legend"]
    )

    scores["categories"]=compare_list(
        reference["categories"],
        generated["categories"]
    )

    if compare_values:

      scores["values"] = compare_list(
          reference["values"],
          generated["values"]
      )

    final=sum(scores.values())/len(scores)

    return scores, final

### 06 - CLIP

In [ ]:
def clip_similarity(reference_path, generated_path):
    """
    Calcula a similaridade visual entre duas imagens
    utilizando o modelo CLIP.
    """

    image1 = preprocess(
        Image.open(reference_path)
    ).unsqueeze(0).to(device)

    image2 = preprocess(
        Image.open(generated_path)
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        emb1 = clip_model.encode_image(image1)
        emb2 = clip_model.encode_image(image2)

        emb1 /= emb1.norm(dim=-1, keepdim=True)
        emb2 /= emb2.norm(dim=-1, keepdim=True)

        similarity = (
            emb1 @ emb2.T
        ).item()

    return similarity

### 08 - Runner

In [ ]:
def get_participants(task, condition):
    """
    Retorna a lista de participantes de uma tarefa e condição.
    """

    folder = os.path.join(
        ROOT,
        task,
        condition
    )

    return sorted([
        p for p in os.listdir(folder)
        if os.path.isdir(os.path.join(folder, p))
    ])

In [ ]:
def get_images(folder):
    """
    Retorna todas as imagens do participante.
    """

    return sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith(VALID_EXTENSIONS)
        and f.startswith("img")
    ])

In [ ]:
def load_references():
    """
    Processa todas as imagens de referência apenas uma vez.
    """

    cache = {}

    for task in TASKS:

        reference_path = os.path.join(
            REF_FOLDER,
            CONFIG[task]["reference"]
        )

        ocr = process_ocr(reference_path)

        cache[task] = parse_chart_components(ocr)

    return cache

In [ ]:
def process_image(reference_path,
                  generated_path,
                  reference_components,
                  compare_values=True):

    participant_ocr = process_ocr(
        generated_path
    )

    participant_components = parse_chart_components(
        participant_ocr
    )

    scores, ocr_score = calculate_text_score(
        reference_components,
        participant_components,
        compare_values=compare_values
    )

    clip_score = clip_similarity(
        reference_path,
        generated_path
    )


    return {

        "ocr": ocr_score,

        "clip": clip_score,

        "components": scores

    }

In [ ]:
def run_experiment(tasks=None):

    reference_cache = load_references()

    results = []

    if tasks is None:
        tasks = TASKS

    for task in tasks:
        reference_path = os.path.join(
            REF_FOLDER,
            CONFIG[task]["reference"]
        )

        reference_components = reference_cache[task]

        compare_values = CONFIG[task]["compare_values"]

        print(f"\n========== {task} ==========\n", flush=True)

        for condition in CONDITIONS:

            print(f"Condição: {condition}", flush=True)

            participants = get_participants(
                task,
                condition
            )

            for participant in participants:

                participant_folder = os.path.join(
                    ROOT,
                    task,
                    condition,
                    participant
                )

                images = get_images(
                    participant_folder
                )

                for image in images:

                    image_path = os.path.join(
                        participant_folder,
                        image
                    )

                    print(
                        f"Processando: {participant} | {image}",
                        flush=True
                    )

                    try:

                        result = process_image(
                            reference_path,
                            image_path,
                            reference_components,
                            compare_values
                        )

                        status = "OK"

                        print("   ✔ OK", flush=True)

                    except Exception as e:

                        status = str(e)

                        result = {
                            "ocr": None,
                            "clip": None
                        }

                        print(f"   ✖ ERRO: {e}", flush=True)

                    results.append({

                        "task": task,

                        "condition": condition,

                        "participant": participant,

                        "trial": image.replace(".png", ""),

                        "image": image,

                        "ocr": result["ocr"],

                        "clip": result["clip"],

                        "status": status

                    })

    return pd.DataFrame(results)

In [ ]:
print(os.listdir(REF_FOLDER))

['barras_comparacao_final.png', 'linhas_comparacao_final.png', 'pontos_comparacao_final.png', 'barras_identificacao_final.png', 'linhas_identificacao_final.png', 'pontos_identificacao_final.png']


In [ ]:
df = run_experiment()

A saída de streaming foi truncada nas últimas 5000 linhas.
   ✔ OK
Processando: P-15 | img07.png
   ✔ OK
Processando: P-15 | img08.png
   ✔ OK
Processando: P-15 | img09.png
   ✔ OK
Processando: P-15 | img10.png
   ✔ OK
Processando: P-16 | img01.png
   ✔ OK
Processando: P-16 | img02.png
   ✔ OK
Processando: P-16 | img03.png
   ✔ OK
Processando: P-16 | img04.png
   ✔ OK
Processando: P-16 | img05.png
   ✔ OK
Processando: P-16 | img06.png
   ✔ OK
Processando: P-16 | img07.png
   ✔ OK
Processando: P-16 | img08.png
   ✔ OK
Processando: P-16 | img09.png
   ✔ OK
Processando: P-16 | img10.png
   ✔ OK
Condição: Few-shot
Processando: P-01 | img01.png
   ✔ OK
Processando: P-01 | img02.png
   ✔ OK
Processando: P-01 | img03.png
   ✔ OK
Processando: P-01 | img04.png
   ✔ OK
Processando: P-01 | img05.png
   ✔ OK
Processando: P-01 | img06.png
   ✔ OK
Processando: P-01 | img07.png
   ✔ OK
Processando: P-01 | img08.png
   ✔ OK
Processando: P-01 | img09.png
   ✔ OK
Processando: P-01 | img10.png
   ✔ OK
Pr

In [ ]:
print(df.head())

     task  condition participant  trial      image       ocr      clip  \
0  B-COMP  Zero-shot        P-01  img01  img01.png  0.864068  0.896745   
1  B-COMP  Zero-shot        P-01  img02  img02.png  0.627045  0.904477   
2  B-COMP  Zero-shot        P-01  img03  img03.png  0.566667  0.912473   
3  B-COMP  Zero-shot        P-01  img04  img04.png  0.954872  0.952457   
4  B-COMP  Zero-shot        P-01  img05  img05.png  0.619887  0.887207   

        ifv status  
0  0.880407     OK  
1  0.765761     OK  
2  0.739570     OK  
3  0.953664     OK  
4  0.753547     OK  


In [ ]:
print(df.shape)

(3120, 9)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3120 entries, 0 to 3119
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   task         3120 non-null   object 
 1   condition    3120 non-null   object 
 2   participant  3120 non-null   object 
 3   trial        3120 non-null   object 
 4   image        3120 non-null   object 
 5   ocr          3120 non-null   float64
 6   clip         3120 non-null   float64
 7   ifv          3120 non-null   float64
 8   status       3120 non-null   object 
dtypes: float64(3), object(6)
memory usage: 219.5+ KB


In [ ]:
df["status"].value_counts()

,count
status,
OK,3120


### 09 - Exportação Excel

In [ ]:
def export_results(df):

    # Cria uma cópia apenas para exportação
    df_export = df.copy()

    # Arredonda apenas na exportação
    df_export[["ocr", "clip"]] = (
        df_export[["ocr", "clip"]]
        .round(2)
    )

    # Média por participante
    summary = (
        df_export
        .groupby(
            ["participant", "condition", "task"]
        )[["ocr", "clip"]]
        .mean()
        .reset_index()
    )

    # Tabelas para o Excel
    ocr_table = (
        summary
        .pivot_table(
            index="participant",
            columns=["condition", "task"],
            values="ocr"
        )
        .round(2)
    )

    clip_table = (
        summary
        .pivot_table(
            index="participant",
            columns=["condition", "task"],
            values="clip"
        )
        .round(2)
    )

    with pd.ExcelWriter("resultado.xlsx") as writer:

        ocr_table.to_excel(writer, sheet_name="OCR")

        clip_table.to_excel(writer, sheet_name="CLIP")

        # Dados completos (uma linha por imagem)
        df_export.to_excel(writer, sheet_name="Dados", index=False)

        # Médias por participante
        summary.to_excel(writer, sheet_name="Resumo", index=False)

    # CSVs
    df_export.to_csv("dados_longos.csv", index=False)
    summary.to_csv("resumo_participante.csv", index=False)

    print("Arquivos exportados com sucesso!")

In [ ]:
export_results(df)

Arquivos exportados com sucesso!


In [ ]:
with open("resumo.txt","w") as f:

    f.write("Similaridade Toolkit\n\n")

    f.write(f"Imagens processadas: {len(df)}\n")

    f.write(f"Participantes: {df['participant'].nunique()}\n")

    f.write(f"Tarefas: {df['task'].nunique()}\n")

    f.write(f"Condições: {df['condition'].nunique()}\n")

    f.write(f"OCR médio: {df['ocr'].mean():.3f}\n")

    f.write(f"CLIP médio: {df['clip'].mean():.3f}\n")